In [1]:
# =========================================
# 02 — Cohort Retention (Olist) | Colab
# Reads orders_fact_light, builds cohort retention (long format), exports to Drive
# =========================================

import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

EXPORT_DIR = "/content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports"
ORDERS_LIGHT_PATH = os.path.join(EXPORT_DIR, "orders_fact_light.csv")

if not os.path.exists(ORDERS_LIGHT_PATH):
    raise FileNotFoundError(f"Could not find: {ORDERS_LIGHT_PATH}")

orders = pd.read_csv(ORDERS_LIGHT_PATH)

Mounted at /content/drive


In [2]:
# Parse timestamp
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

In [3]:
# Keep only valid rows
orders = orders.dropna(subset=["customer_unique_id", "order_purchase_timestamp"]).copy()

In [4]:
# Create order_month (YYYY-MM)
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

In [5]:
# -------------------------------------------------
# 1) First purchase month per customer = cohort_month
# -------------------------------------------------
first_purchase = (
    orders.groupby("customer_unique_id", as_index=False)["order_purchase_timestamp"]
    .min()
    .rename(columns={"order_purchase_timestamp": "first_purchase_ts"})
)
first_purchase["cohort_month"] = first_purchase["first_purchase_ts"].dt.to_period("M").astype(str)

orders = orders.merge(first_purchase[["customer_unique_id", "cohort_month"]], on="customer_unique_id", how="left")

In [7]:
# -------------------------------------------------
# 2) month_index = months between order_month and cohort_month (robust)
# -------------------------------------------------

# order_period as period
orders["order_period"] = orders["order_purchase_timestamp"].dt.to_period("M")

# cohort_period as period (from cohort_month string like '2017-01')
orders["cohort_period"] = pd.to_datetime(orders["cohort_month"] + "-01").dt.to_period("M")

# month_index = (year diff * 12) + (month diff)
orders["month_index"] = (
    (orders["order_period"].dt.year - orders["cohort_period"].dt.year) * 12
    + (orders["order_period"].dt.month - orders["cohort_period"].dt.month)
).astype(int)

# safety
orders = orders[orders["month_index"] >= 0].copy()

In [8]:
# Keep non-negative indices only (should already be, but safe)
orders = orders[orders["month_index"] >= 0].copy()

In [9]:
# -------------------------------------------------
# 3) Cohort size (month_index=0) and active customers by index
# -------------------------------------------------
active = (
    orders.groupby(["cohort_month", "month_index"])["customer_unique_id"]
    .nunique()
    .reset_index(name="active_customers")
)

cohort_size = (
    active[active["month_index"] == 0][["cohort_month", "active_customers"]]
    .rename(columns={"active_customers": "cohort_size"})
)

cohort_retention = active.merge(cohort_size, on="cohort_month", how="left")
cohort_retention["retention_rate"] = cohort_retention["active_customers"] / cohort_retention["cohort_size"]

In [10]:
# Optional: cap to first 12 months for a clean dashboard
cohort_retention_12 = cohort_retention[cohort_retention["month_index"] <= 12].copy()

In [11]:
# -------------------------------------------------
# 4) Export (Power BI-friendly long format)
# -------------------------------------------------
out_path = os.path.join(EXPORT_DIR, "cohort_retention_long.csv")
cohort_retention_12.to_csv(out_path, index=False)

print("✅ Exported:", out_path)
print("\nPreview:")
cohort_retention_12.sort_values(["cohort_month", "month_index"]).head(20)

✅ Exported: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports/cohort_retention_long.csv

Preview:


,cohort_month,month_index,active_customers,cohort_size,retention_rate
0,2016-09,0,1,1,1.000000
1,2016-10,0,262,262,1.000000
2,2016-10,6,1,262,0.003817
3,2016-10,9,1,262,0.003817
4,2016-10,11,1,262,0.003817
10,2016-12,0,1,1,1.000000
11,2016-12,1,1,1,1.000000
12,2017-01,0,717,717,1.000000
13,2017-01,1,2,717,0.002789
14,2017-01,2,2,717,0.002789
